In [1]:
import pandas as pd
import numpy as np
import openpyxl
import pytest

print("개발 환경 준비 완료")

ModuleNotFoundError: No module named 'pytest'

In [2]:
%pip install pytest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pytest]2m2/3 [pytest]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import openpyxl
import pytest

print("개발 환경 준비 완료")

개발 환경 준비 완료


In [4]:
from pathlib import Path

BASE_DIR = Path.cwd()

folders = [
    "data",
    "src",
    "tests",
    "outputs",
    "docs",
    "notebooks"
]

for folder in folders:
    (BASE_DIR / folder).mkdir(exist_ok=True)

print(BASE_DIR)
print("프로젝트 폴더 생성 완료")

/Users/kosunjin/Desktop/TaxDataValidator
프로젝트 폴더 생성 완료


In [5]:
TAX_RATES = {
    "INTEREST": 14,
    "BUSINESS": 3,
    "OTHER": 20
}

In [6]:
TAX_RATES["BUSINESS"]

3

In [7]:
def truncate_10(amount):
    return (amount // 10) * 10

In [8]:
truncate_10(1238)

1230

In [9]:

from decimal import Decimal
from numbers import Integral


def calculate_tax(
    income_type,
    gross_amount,
    residency="RESIDENT"
):

    # 1. 입력 데이터 검증
    if isinstance(gross_amount, bool) or not isinstance(
        gross_amount, Integral
    ):
        raise ValueError("지급액은 정수여야 합니다.")

    if gross_amount <= 0:
        raise ValueError("지급액은 양수여야 합니다.")

    # 2. 프로젝트 적용 범위 확인
    if residency != "RESIDENT":
        raise ValueError("거주자 여부 추가 검토 필요")

    if income_type not in TAX_RATES:
        raise ValueError("지원하지 않는 소득 유형")

    gross = Decimal(int(gross_amount))
    expense = Decimal("0")
    reason = "일반 원천징수"

    # 3. 소득 유형별 원천징수 기준금액
    if income_type == "OTHER":
        expense = gross * Decimal("0.60")

    tax_base = gross - expense

    # 4. 과세최저한 판단
    if income_type == "OTHER" and tax_base <= 50000:
        raw_tax = Decimal("0")
        reason = "기타소득 과세최저한"

    else:
        rate = Decimal(TAX_RATES[income_type]) / 100
        raw_tax = tax_base * rate

        # 5. 소액부징수 예외 확인
        if (
            income_type in ["INTEREST", "BUSINESS"]
            and raw_tax < 1000
        ):
            reason = "소액부징수 적용 제외"

    # 6. 소득세 및 지방소득세 계산
    income_tax = int(truncate_10(raw_tax))
    local_tax = int(
        truncate_10(Decimal(income_tax) * Decimal("0.1"))
    )

    # 7. 실지급액 계산
    total_tax = income_tax + local_tax
    net_amount = int(gross) - total_tax

    # 8. 결과 반환
    return {
        "gross_amount": int(gross),
        "expense_amount": expense,
        "tax_base": tax_base,
        "income_tax": income_tax,
        "local_tax": local_tax,
        "total_tax": total_tax,
        "net_amount": net_amount,
        "reason": reason
    }


In [10]:
print(0.1 + 0.2)

0.30000000000000004


In [11]:
raise ValueError("지원하지 않는 소득 유형")

ValueError: 지원하지 않는 소득 유형

In [12]:
calculate_tax("BUSINESS", 1000000)

{'gross_amount': 1000000,
 'expense_amount': Decimal('0'),
 'tax_base': Decimal('1000000'),
 'income_tax': 30000,
 'local_tax': 3000,
 'total_tax': 33000,
 'net_amount': 967000,
 'reason': '일반 원천징수'}

In [13]:
import pandas as pd

cases = [
    ("INTEREST", 1000000),
    ("INTEREST", 5000),
    ("BUSINESS", 1000000),
    ("BUSINESS", 30000),
    ("OTHER", 1000000),
    ("OTHER", 125000),
    ("OTHER", 125005)
]

results = []

for income_type, gross_amount in cases:

    result = calculate_tax(
        income_type,
        gross_amount
    )

    result["income_type"] = income_type

    results.append(result)

df = pd.DataFrame(results)

df[[
    "income_type",
    "gross_amount",
    "income_tax",
    "local_tax",
    "net_amount",
    "reason"
]]

,income_type,gross_amount,income_tax,local_tax,net_amount,reason
0,INTEREST,1000000,140000,14000,846000,일반 원천징수
1,INTEREST,5000,700,70,4230,소액부징수 적용 제외
2,BUSINESS,1000000,30000,3000,967000,일반 원천징수
3,BUSINESS,30000,900,90,29010,소액부징수 적용 제외
4,OTHER,1000000,80000,8000,912000,일반 원천징수
5,OTHER,125000,0,0,125000,기타소득 과세최저한
6,OTHER,125005,10000,1000,114005,일반 원천징수


In [14]:
from pathlib import Path

BASE_DIR = Path.cwd()

assert BASE_DIR.name == "TaxDataValidator", BASE_DIR

for folder in ["data", "src", "tests", "outputs", "docs", "notebooks"]:
    (BASE_DIR / folder).mkdir(exist_ok=True)

(BASE_DIR / "src" / "__init__.py").touch()

print(BASE_DIR)

/Users/kosunjin/Desktop/TaxDataValidator


In [15]:
%%writefile src/tax_rules.py
from decimal import Decimal
from numbers import Integral

TAX_RATES = {
    "INTEREST": 14,
    "BUSINESS": 3,
    "OTHER": 20
}

SUBTYPES = {
    "INTEREST": "BANK_DEPOSIT",
    "BUSINESS": "RECURRING_SERVICE",
    "OTHER": "ONE_TIME_LECTURE"
}


class ReviewRequired(ValueError):
    pass


def truncate_10(amount):
    return int((amount // 10) * 10)


def calculate_tax(
    income_type,
    gross_amount,
    residency="RESIDENT",
    income_subtype=None
):

    if isinstance(gross_amount, bool) or not isinstance(
        gross_amount, Integral
    ):
        raise ValueError("지급액은 정수여야 합니다.")

    if gross_amount <= 0:
        raise ValueError("지급액은 양수여야 합니다.")

    if residency != "RESIDENT":
        raise ReviewRequired("거주자 여부 추가 검토 필요")

    if income_type not in TAX_RATES:
        raise ReviewRequired("지원하지 않는 소득 유형")

    if income_subtype != SUBTYPES[income_type]:
        raise ReviewRequired("지원하지 않는 세부 소득 유형")

    gross = Decimal(int(gross_amount))
    expense = Decimal("0")
    reason = "일반 원천징수"

    if income_type == "OTHER":

        # 원 미만 필요경비 발생 건은 MVP 검토 대상으로 분리
        if gross_amount % 5 != 0:
            raise ReviewRequired("필요경비 원 미만 단수 처리 확인")

        expense = gross * Decimal("0.60")

    tax_base = gross - expense

    if income_type == "OTHER" and tax_base <= 50000:
        raw_tax = Decimal("0")
        reason = "기타소득 과세최저한"

    else:
        rate = Decimal(TAX_RATES[income_type]) / 100
        raw_tax = tax_base * rate

        if income_type in ["INTEREST", "BUSINESS"] and raw_tax < 1000:
            reason = "소액부징수 적용 제외"

    # 학습용 가정: 지급 건별 10원 미만 절사
    income_tax = truncate_10(raw_tax)
    local_tax = truncate_10(Decimal(income_tax) / 10)

    total_tax = income_tax + local_tax
    net_amount = int(gross) - total_tax

    return {
        "gross_amount": int(gross),
        "expense_amount": int(expense),
        "tax_base": int(tax_base),
        "income_tax": income_tax,
        "local_tax": local_tax,
        "total_tax": total_tax,
        "net_amount": net_amount,
        "reason": reason
    }

Writing src/tax_rules.py


In [16]:
from src.tax_rules import calculate_tax

calculate_tax(
    "BUSINESS",
    1000000,
    income_subtype="RECURRING_SERVICE"
)

{'gross_amount': 1000000,
 'expense_amount': 0,
 'tax_base': 1000000,
 'income_tax': 30000,
 'local_tax': 3000,
 'total_tax': 33000,
 'net_amount': 967000,
 'reason': '일반 원천징수'}

In [17]:
import pandas as pd
from src.tax_rules import SUBTYPES

cases = [
    ("INTEREST", 1000000),
    ("INTEREST", 5000),
    ("BUSINESS", 1000000),
    ("BUSINESS", 30000),
    ("OTHER", 1000000),
    ("OTHER", 125000),
    ("OTHER", 125005)
]

results = []

for kind, gross in cases:
    result = calculate_tax(
        kind,
        gross,
        income_subtype=SUBTYPES[kind]
    )
    result["income_type"] = kind
    results.append(result)

pd.DataFrame(results)

,gross_amount,expense_amount,tax_base,income_tax,local_tax,total_tax,net_amount,reason,income_type
0,1000000,0,1000000,140000,14000,154000,846000,일반 원천징수,INTEREST
1,5000,0,5000,700,70,770,4230,소액부징수 적용 제외,INTEREST
2,1000000,0,1000000,30000,3000,33000,967000,일반 원천징수,BUSINESS
3,30000,0,30000,900,90,990,29010,소액부징수 적용 제외,BUSINESS
4,1000000,600000,400000,80000,8000,88000,912000,일반 원천징수,OTHER
5,125000,75000,50000,0,0,0,125000,기타소득 과세최저한,OTHER
6,125005,75003,50002,10000,1000,11000,114005,일반 원천징수,OTHER


In [18]:
%%writefile src/generate_data.py
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path(__file__).resolve().parents[1]
DATA_DIR = BASE_DIR / "data"

# 수기로 계산한 정상 거래 사례
# 소득유형, 세부유형, 지급액, 필요경비, 소득세, 지방소득세

FIXTURES = [
    ("INTEREST", "BANK_DEPOSIT", 1000000, 0, 140000, 14000),
    ("INTEREST", "BANK_DEPOSIT", 5000, 0, 700, 70),
    ("INTEREST", "BANK_DEPOSIT", 200000, 0, 28000, 2800),
    ("INTEREST", "BANK_DEPOSIT", 500000, 0, 70000, 7000),
    ("BUSINESS", "RECURRING_SERVICE", 1000000, 0, 30000, 3000),
    ("BUSINESS", "RECURRING_SERVICE", 30000, 0, 900, 90),
    ("BUSINESS", "RECURRING_SERVICE", 800000, 0, 24000, 2400),
    ("BUSINESS", "RECURRING_SERVICE", 2000000, 0, 60000, 6000),
    ("OTHER", "ONE_TIME_LECTURE", 125000, 75000, 0, 0),
    ("OTHER", "ONE_TIME_LECTURE", 125005, 75003, 10000, 1000),
    ("OTHER", "ONE_TIME_LECTURE", 500000, 300000, 40000, 4000),
    ("OTHER", "ONE_TIME_LECTURE", 1000000, 600000, 80000, 8000)
]

DETAIL_COLUMNS = [
    "payment_id",
    "recipient_id",
    "income_type",
    "accrual_month",
    "payment_date",
    "gross_amount",
    "income_tax",
    "local_tax"
]


def make_summary(details):
    return (
        details.groupby("income_type", as_index=False)
        .agg(
            payment_count=("payment_id", "size"),
            gross_amount=("gross_amount", "sum"),
            income_tax=("income_tax", "sum"),
            local_tax=("local_tax", "sum")
        )
    )


def generate_data(seed=42):

    DATA_DIR.mkdir(exist_ok=True)

    rng = np.random.default_rng(seed)
    rows = []

    fixed_indices = [0, 4, 10, 2, 4, 6, 8, 5, 11, 3, 0, 4]

    for i in range(199):

        fixture_index = (
            fixed_indices[i]
            if i < len(fixed_indices)
            else int(rng.integers(0, len(FIXTURES)))
        )

        kind, subtype, gross, expense, tax, local = (
            FIXTURES[fixture_index]
        )

        rows.append({
            "row_id": f"R{i + 1:03d}",
            "payment_id": f"P{i + 1:03d}",
            "recipient_id": f"C{rng.integers(1, 101):03d}",
            "residency": "RESIDENT",
            "income_type": kind,
            "income_subtype": subtype,
            "accrual_month": "2026-10",
            "payment_date": f"2026-10-{rng.integers(1, 32):02d}",
            "gross_amount": gross,
            "expense_amount": expense,
            "income_tax": tax,
            "local_tax": local,
            "net_amount": gross - tax - local
        })

    clean = pd.DataFrame(rows)
    payments = clean.copy()

    # 세액 오류 삽입
    payments.loc[4, "income_tax"] += 1000
    payments.loc[4, "net_amount"] -= 1000

    payments.loc[5, "local_tax"] += 100
    payments.loc[5, "net_amount"] -= 100

    payments.loc[6, "income_tax"] = 10000
    payments.loc[6, "net_amount"] -= 10000

    payments.loc[7, ["income_tax", "local_tax"]] = 0
    payments.loc[7, "net_amount"] = payments.loc[7, "gross_amount"]

    payments.loc[8, "net_amount"] += 500

    # 공통 가상 지급 기록을 기반으로 별도 신고자료 구성
    details = payments[DETAIL_COLUMNS].copy()
    summary = make_summary(details)

    # 원장 데이터 오류
    payments.loc[0, "recipient_id"] = ""
    payments.loc[1, "gross_amount"] = -100000
    payments.loc[2, "payment_date"] = "2026-10-32"
    payments.loc[3, "residency"] = "NON_RESIDENT"

    # 중복 거래 삽입
    duplicate = payments.iloc[[9]].copy()
    duplicate.loc[:, "row_id"] = "R200"

    payments = pd.concat(
        [payments, duplicate],
        ignore_index=True
    )

    # 신고자료 누락
    details = details[details["payment_id"] != "P011"].copy()

    # 신고자료 금액 불일치
    details.loc[
        details["payment_id"] == "P012",
        "gross_amount"
    ] += 10000

    # 신고자료에만 존재하는 거래
    extra = details.iloc[[6]].copy()

    extra.loc[:, "payment_id"] = "P999"
    extra.loc[:, "recipient_id"] = "C999"

    details = pd.concat([details, extra], ignore_index=True)

    # 오류 정답표: 검증 함수와 독립적으로 지정
    truth = [
        ("PAYMENT", "R001", "DATA-01"),
        ("PAYMENT", "R002", "DATA-03"),
        ("PAYMENT", "R003", "DATA-04"),
        ("PAYMENT", "R004", "TAX-01"),
        ("PAYMENT", "R005", "TAX-02"),
        ("PAYMENT", "R006", "TAX-03"),
        ("PAYMENT", "R007", "TAX-02"),
        ("PAYMENT", "R007", "TAX-04"),
        ("PAYMENT", "R008", "TAX-02"),
        ("PAYMENT", "R008", "TAX-03"),
        ("PAYMENT", "R008", "TAX-05"),
        ("PAYMENT", "R009", "TAX-06"),
        ("PAYMENT", "R010", "DATA-02"),
        ("PAYMENT", "R200", "DATA-02"),
        ("DETAIL", "P011", "REC-01"),
        ("DETAIL", "P012", "REC-03"),
        ("DETAIL", "P999", "REC-02"),
        ("SUMMARY", "INTEREST", "REC-04"),
        ("SUMMARY", "BUSINESS", "REC-04"),
        ("SUMMARY", "OTHER", "REC-04")
    ]

    truth = pd.DataFrame(
        truth,
        columns=["scope", "key", "code"]
    )

    for filename, frame in [
        ("payments.csv", payments),
        ("filing_details.csv", details),
        ("filing_summary.csv", summary),
        ("ground_truth.csv", truth)
    ]:
        frame.to_csv(
            DATA_DIR / filename,
            index=False,
            encoding="utf-8-sig"
        )

    return payments, details, summary, truth


if __name__ == "__main__":
    payments, details, summary, truth = generate_data()
    print("지급자료:", len(payments))
    print("신고 기초자료:", len(details))
    print("정답표 검증 항목:", len(truth))

Writing src/generate_data.py


In [19]:
from src.generate_data import generate_data

payments, details, summary, truth = generate_data()

print("지급자료:", payments.shape)
print("신고 기초자료:", details.shape)
print("집계자료:", summary.shape)
print("오류 정답표:", truth.shape)

display(payments.head(12))
display(summary)

지급자료: (200, 13)
신고 기초자료: (199, 8)
집계자료: (3, 5)
오류 정답표: (20, 3)


,row_id,payment_id,recipient_id,residency,income_type,income_subtype,accrual_month,payment_date,gross_amount,expense_amount,income_tax,local_tax,net_amount
0,R001,P001,,RESIDENT,INTEREST,BANK_DEPOSIT,2026-10,2026-10-24,1000000,0,140000,14000,846000
1,R002,P002,C066,RESIDENT,BUSINESS,RECURRING_SERVICE,2026-10,2026-10-14,-100000,0,30000,3000,967000
2,R003,P003,C044,RESIDENT,OTHER,ONE_TIME_LECTURE,2026-10,2026-10-32,500000,300000,40000,4000,456000
3,R004,P004,C009,NON_RESIDENT,INTEREST,BANK_DEPOSIT,2026-10,2026-10-22,200000,0,28000,2800,169200
4,R005,P005,C021,RESIDENT,BUSINESS,RECURRING_SERVICE,2026-10,2026-10-03,1000000,0,31000,3000,966000
5,R006,P006,C053,RESIDENT,BUSINESS,RECURRING_SERVICE,2026-10,2026-10-31,800000,0,24000,2500,773500
6,R007,P007,C074,RESIDENT,OTHER,ONE_TIME_LECTURE,2026-10,2026-10-24,125000,75000,10000,0,115000
7,R008,P008,C072,RESIDENT,BUSINESS,RECURRING_SERVICE,2026-10,2026-10-25,30000,0,0,0,30000
8,R009,P009,C052,RESIDENT,OTHER,ONE_TIME_LECTURE,2026-10,2026-10-04,1000000,600000,80000,8000,912500
9,R010,P010,C084,RESIDENT,INTEREST,BANK_DEPOSIT,2026-10,2026-10-14,500000,0,70000,7000,423000


,income_type,payment_count,gross_amount,income_tax,local_tax
0,BUSINESS,58,48970000,1469200,146920
1,INTEREST,71,27615000,3866100,386610
2,OTHER,70,36250090,2790000,278000


In [20]:
%%writefile src/validator.py
from datetime import date
from decimal import Decimal, InvalidOperation

import pandas as pd

from src.tax_rules import calculate_tax, ReviewRequired

TARGET_MONTH = "2026-10"

REQUIRED = [
    "row_id",
    "payment_id",
    "recipient_id",
    "residency",
    "income_type",
    "income_subtype",
    "accrual_month",
    "payment_date",
    "gross_amount",
    "expense_amount",
    "income_tax",
    "local_tax",
    "net_amount"
]

AMOUNT_COLUMNS = [
    "gross_amount",
    "expense_amount",
    "income_tax",
    "local_tax",
    "net_amount"
]

EVENT_COLUMNS = [
    "scope", "key", "code", "status",
    "message", "expected", "actual"
]


def is_missing(value):
    return bool(pd.isna(value)) or str(value).strip() == ""


def as_integer(value):
    try:
        number = Decimal(str(value))

        if not number.is_finite():
            return None

        if number != number.to_integral_value():
            return None

        return int(number)

    except (InvalidOperation, ValueError, TypeError):
        return None


def event(scope, key, code, message,
          status="ERROR", expected=None, actual=None):

    return {
        "scope": scope,
        "key": key,
        "code": code,
        "status": status,
        "message": message,
        "expected": expected,
        "actual": actual
    }


def validate_payments(df):

    absent = set(REQUIRED) - set(df.columns)

    if absent:
        raise ValueError(f"필수 컬럼 누락: {sorted(absent)}")

    transactions = []
    events = []

    duplicate_mask = df["payment_id"].duplicated(keep=False)
    duplicate_rows = df["row_id"].duplicated(keep=False)

    for pos, (_, row) in enumerate(df.iterrows()):

        record = row.to_dict()
        key = str(row["row_id"]) if not is_missing(
            row["row_id"]
        ) else f"ROW_{pos + 1}"

        issues = []

        def add(code, message, status="ERROR",
                expected=None, actual=None):

            issues.append(
                event(
                    "PAYMENT", key, code, message,
                    status, expected, actual
                )
            )

        # DATA-01: 필수값 누락
        missing = [
            col for col in REQUIRED
            if is_missing(row[col])
        ]

        if missing:
            add("DATA-01", "필수값 누락: " + ", ".join(missing))

        # DATA-02: 거래 ID 또는 행 ID 중복
        if duplicate_mask.iloc[pos] or duplicate_rows.iloc[pos]:
            add("DATA-02", "거래 식별번호 중복")

        # DATA-03: 금액 오류
        amounts = {}

        for col in AMOUNT_COLUMNS:

            if is_missing(row[col]):
                continue

            value = as_integer(row[col])

            if (
                value is None
                or value < 0
                or (col == "gross_amount" and value == 0)
            ):
                add(
                    "DATA-03",
                    f"잘못된 금액: {col}",
                    actual=str(row[col])
                )
            else:
                amounts[col] = value

        # DATA-04: 날짜 검증
        if not is_missing(row["payment_date"]):

            try:
                payment_date = date.fromisoformat(
                    str(row["payment_date"])
                )

                if payment_date.strftime("%Y-%m") != TARGET_MONTH:
                    add("DATA-04", "신고 대상 지급월 불일치")

            except ValueError:
                add("DATA-04", "유효하지 않은 지급일자")

        if not is_missing(row["accrual_month"]):
            try:
                month = str(row["accrual_month"])
                parsed = date.fromisoformat(month + "-01")

                if parsed.strftime("%Y-%m") != month:
                    raise ValueError

            except ValueError:
                add("DATA-04", "유효하지 않은 귀속연월")

        structural_error = any(
            issue["code"].startswith("DATA")
            for issue in issues
        )

        record["reconciliation_eligible"] = not structural_error
        record["tax_calculated"] = False

        for col in [
            "expected_income_tax",
            "expected_local_tax",
            "expected_expense_amount",
            "income_tax_diff",
            "local_tax_diff",
            "expected_net_amount"
        ]:
            record[col] = None

        if not structural_error:

            gross = amounts["gross_amount"]
            recorded_tax = amounts["income_tax"]
            recorded_local = amounts["local_tax"]

            # TAX-06: 실지급액 대사
            expected_net = gross - recorded_tax - recorded_local

            if (
                amounts["net_amount"] != expected_net
                or expected_net < 0
            ):
                add(
                    "TAX-06",
                    "실지급액과 기록 세액 불일치",
                    expected=expected_net,
                    actual=amounts["net_amount"]
                )

            try:
                result = calculate_tax(
                    row["income_type"],
                    gross,
                    row["residency"],
                    row["income_subtype"]
                )

                expected_tax = result["income_tax"]
                expected_local = result["local_tax"]

                record.update({
                    "tax_calculated": True,
                    "expected_income_tax": expected_tax,
                    "expected_local_tax": expected_local,
                    "expected_expense_amount": result["expense_amount"],
                    "income_tax_diff": recorded_tax - expected_tax,
                    "local_tax_diff": recorded_local - expected_local,
                    "expected_net_amount": result["net_amount"],
                    "tax_reason": result["reason"]
                })

                # TAX-02: 소득세 불일치
                if recorded_tax != expected_tax:
                    add(
                        "TAX-02", "소득세 불일치",
                        expected=expected_tax,
                        actual=recorded_tax
                    )

                # TAX-03: 지방소득세 불일치
                if recorded_local != expected_local:
                    add(
                        "TAX-03", "지방소득세 불일치",
                        expected=expected_local,
                        actual=recorded_local
                    )

                # TAX-04: 과세최저한 미적용
                if (
                    result["reason"] == "기타소득 과세최저한"
                    and (recorded_tax > 0 or recorded_local > 0)
                ):
                    add("TAX-04", "기타소득 과세최저한 미적용")

                # TAX-05: 소액부징수 예외 미반영
                if (
                    result["reason"] == "소액부징수 적용 제외"
                    and expected_tax > 0
                    and recorded_tax == 0
                ):
                    add("TAX-05", "소액부징수 적용 제외 규정 확인")

                # TAX-07: 필요경비 검토
                if amounts["expense_amount"] != result["expense_amount"]:
                    add(
                        "TAX-07",
                        "필요경비 적용 근거 추가 확인",
                        status="REVIEW",
                        expected=result["expense_amount"],
                        actual=amounts["expense_amount"]
                    )

                    # 필요경비가 다르면 확정 세액 차이에서 제외
                    record["tax_calculated"] = False
                    record["income_tax_diff"] = None
                    record["local_tax_diff"] = None

                    for issue in issues:
                        if issue["code"] in [
                            "TAX-02", "TAX-03", "TAX-04"
                        ]:
                            issue["status"] = "REVIEW"

            except ReviewRequired as exc:
                add("TAX-01", str(exc), status="REVIEW")

        statuses = [issue["status"] for issue in issues]

        if "ERROR" in statuses:
            record["status"] = "ERROR"
        elif "REVIEW" in statuses:
            record["status"] = "REVIEW"
        else:
            record["status"] = "PASS"

        record["error_codes"] = ", ".join(
            sorted({issue["code"] for issue in issues})
        )

        transactions.append(record)
        events.extend(issues)

    return (
        pd.DataFrame(transactions),
        pd.DataFrame(events, columns=EVENT_COLUMNS)
    )

Writing src/validator.py


In [21]:
from src.validator import validate_payments

transactions, payment_events = validate_payments(payments)

display(transactions["status"].value_counts())
display(payment_events)

status
PASS      189
ERROR      10
REVIEW      1
Name: count, dtype: int64

,scope,key,code,status,message,expected,actual
0,PAYMENT,R001,DATA-01,ERROR,필수값 누락: recipient_id,NaN,None
1,PAYMENT,R002,DATA-03,ERROR,잘못된 금액: gross_amount,NaN,-100000
2,PAYMENT,R003,DATA-04,ERROR,유효하지 않은 지급일자,NaN,None
3,PAYMENT,R004,TAX-01,REVIEW,거주자 여부 추가 검토 필요,NaN,None
4,PAYMENT,R005,TAX-02,ERROR,소득세 불일치,30000.0,31000
5,PAYMENT,R006,TAX-03,ERROR,지방소득세 불일치,2400.0,2500
6,PAYMENT,R007,TAX-02,ERROR,소득세 불일치,0.0,10000
7,PAYMENT,R007,TAX-04,ERROR,기타소득 과세최저한 미적용,NaN,None
8,PAYMENT,R008,TAX-02,ERROR,소득세 불일치,900.0,0
9,PAYMENT,R008,TAX-03,ERROR,지방소득세 불일치,90.0,0


In [22]:
%%writefile src/reconciliation.py
import pandas as pd

from src.generate_data import DETAIL_COLUMNS, make_summary
from src.validator import as_integer, is_missing, event, EVENT_COLUMNS

COMPARE_COLUMNS = [
    col for col in DETAIL_COLUMNS if col != "payment_id"
]

DETAIL_AMOUNTS = [
    "gross_amount", "income_tax", "local_tax"
]

SUMMARY_COLUMNS = [
    "payment_count", "gross_amount", "income_tax", "local_tax"
]


def reconcile(transactions, details, summary):

    events = []
    detail_results = []
    summary_results = []

    if set(DETAIL_COLUMNS) - set(details.columns):
        raise ValueError("신고 기초자료 필수 컬럼 누락")

    if set(["income_type"] + SUMMARY_COLUMNS) - set(summary.columns):
        raise ValueError("집계자료 필수 컬럼 누락")

    details = details.copy()

    blocked_ids = set(
        transactions.loc[
            ~transactions["reconciliation_eligible"],
            "payment_id"
        ]
    )

    duplicate_mask = details["payment_id"].duplicated(keep=False)
    invalid_positions = set()

    # 신고 기초자료 자체의 정합성 검증
    for pos, (_, row) in enumerate(details.iterrows()):

        key = str(row["payment_id"])

        if any(is_missing(row[col]) for col in DETAIL_COLUMNS):

            events.append(event(
                "DETAIL", key, "DATA-01", "신고자료 필수값 누락"
            ))

            invalid_positions.add(pos)

        if duplicate_mask.iloc[pos]:

            events.append(event(
                "DETAIL", key, "DATA-02", "신고자료 거래 중복"
            ))

            invalid_positions.add(pos)

        invalid_amounts = [
            col for col in DETAIL_AMOUNTS
            if as_integer(row[col]) is None
            or as_integer(row[col]) < 0
            or (
                col == "gross_amount"
                and as_integer(row[col]) == 0
            )
        ]

        if invalid_amounts:

            events.append(event(
                "DETAIL", key, "DATA-03",
                "신고자료 금액 오류: " + ", ".join(invalid_amounts)
            ))

            invalid_positions.add(pos)

    if invalid_positions:
        blocked_ids.update(
            details.iloc[sorted(invalid_positions)]["payment_id"]
        )

    for key in sorted(map(str, blocked_ids)):
        detail_results.append({
            "payment_id": key,
            "status": "BLOCKED",
            "message": "원장 또는 신고자료의 정합성 오류로 대사 보류"
        })

    left = transactions.loc[
        transactions["reconciliation_eligible"]
        & ~transactions["payment_id"].isin(blocked_ids),
        DETAIL_COLUMNS
    ].copy()

    right = details.loc[
        ~details["payment_id"].isin(blocked_ids),
        DETAIL_COLUMNS
    ].copy()

    for col in DETAIL_AMOUNTS:
        left[col] = left[col].map(as_integer)
        right[col] = right[col].map(as_integer)

    merged = left.merge(
        right,
        on="payment_id",
        how="outer",
        suffixes=("_ledger", "_filing"),
        indicator=True,
        validate="one_to_one"
    )

    for _, row in merged.iterrows():

        key = row["payment_id"]
        record = row.to_dict()

        if row["_merge"] == "left_only":

            code = "REC-01"
            message = "신고 기초자료 누락"

        elif row["_merge"] == "right_only":

            code = "REC-02"
            message = "지급 원장에 없는 신고자료"

        else:

            differences = [
                col for col in COMPARE_COLUMNS
                if row[f"{col}_ledger"] != row[f"{col}_filing"]
            ]

            if differences:
                code = "REC-03"
                message = "거래 불일치: " + ", ".join(differences)
            else:
                code = None
                message = "대사 일치"

        record["status"] = "ERROR" if code else "PASS"
        record["code"] = code
        record["message"] = message

        detail_results.append(record)

        if code:
            events.append(event(
                "DETAIL", key, code, message
            ))

    # 집계자료 자체의 정합성 검증
    invalid_summary = (
        summary["income_type"].duplicated().any()
        or any(
            is_missing(value)
            for value in summary["income_type"]
        )
        or any(
            as_integer(value) is None or as_integer(value) < 0
            for col in SUMMARY_COLUMNS
            for value in summary[col]
        )
    )

    # 신고 상세자료 오류가 있으면 집계 대사를 보류
    if invalid_positions or invalid_summary:

        events.append(event(
            "SUMMARY", "ALL", "REC-04",
            "상세 또는 집계자료 정합성 오류로 집계 대사 보류",
            status="REVIEW"
        ))

        summary_results.append({
            "income_type": "ALL",
            "status": "BLOCKED",
            "message": "집계자료 정합성 확인 필요"
        })

    else:

        for col in DETAIL_AMOUNTS:
            details[col] = details[col].map(as_integer)

        summary = summary.copy()

        for col in SUMMARY_COLUMNS:
            summary[col] = summary[col].map(as_integer)

        expected_summary = make_summary(details)

        merged_summary = expected_summary.merge(
            summary,
            on="income_type",
            how="outer",
            suffixes=("_detail", "_filing"),
            indicator=True,
            validate="one_to_one"
        )

        for _, row in merged_summary.iterrows():

            key = row["income_type"]
            record = row.to_dict()

            different = []

            for col in SUMMARY_COLUMNS:

                expected = row[f"{col}_detail"]
                actual = row[f"{col}_filing"]

                if pd.isna(expected) or pd.isna(actual):
                    different.append(col)
                    record[f"{col}_diff"] = None
                else:
                    record[f"{col}_diff"] = actual - expected

                    if actual != expected:
                        different.append(col)

            record["status"] = "ERROR" if different else "PASS"

            record["message"] = (
                "집계 불일치: " + ", ".join(different)
                if different
                else "집계 일치"
            )

            summary_results.append(record)

            if different:

                events.append(event(
                    "SUMMARY",
                    key,
                    "REC-04",
                    record["message"]
                ))

    return (
        pd.DataFrame(detail_results),
        pd.DataFrame(summary_results),
        pd.DataFrame(events, columns=EVENT_COLUMNS)
    )

Writing src/reconciliation.py


In [23]:
from src.reconciliation import reconcile

detail_result, summary_result, rec_events = reconcile(
    transactions,
    details,
    summary
)

display(rec_events)

display(
    detail_result[
        detail_result["status"] != "PASS"
    ][["payment_id", "status", "message"]]
)

display(summary_result)

,scope,key,code,status,message,expected,actual
0,DETAIL,P011,REC-01,ERROR,신고 기초자료 누락,None,None
1,DETAIL,P012,REC-03,ERROR,거래 불일치: gross_amount,None,None
2,DETAIL,P999,REC-02,ERROR,지급 원장에 없는 신고자료,None,None
3,SUMMARY,BUSINESS,REC-04,ERROR,집계 불일치: gross_amount,None,None
4,SUMMARY,INTEREST,REC-04,ERROR,"집계 불일치: payment_count, gross_amount, income_ta...",None,None
5,SUMMARY,OTHER,REC-04,ERROR,"집계 불일치: payment_count, gross_amount, income_tax",None,None


,payment_id,status,message
0,P001,BLOCKED,원장 또는 신고자료의 정합성 오류로 대사 보류
1,P002,BLOCKED,원장 또는 신고자료의 정합성 오류로 대사 보류
2,P003,BLOCKED,원장 또는 신고자료의 정합성 오류로 대사 보류
3,P010,BLOCKED,원장 또는 신고자료의 정합성 오류로 대사 보류
10,P011,ERROR,신고 기초자료 누락
11,P012,ERROR,거래 불일치: gross_amount
199,P999,ERROR,지급 원장에 없는 신고자료


,income_type,payment_count_detail,gross_amount_detail,income_tax_detail,local_tax_detail,payment_count_filing,gross_amount_filing,income_tax_filing,local_tax_filing,_merge,payment_count_diff,gross_amount_diff,income_tax_diff,local_tax_diff,status,message
0,BUSINESS,58,48980000,1469200,146920,58,48970000,1469200,146920,both,0,-10000,0,0,ERROR,집계 불일치: gross_amount
1,INTEREST,70,26615000,3726100,372610,71,27615000,3866100,386610,both,1,1000000,140000,14000,ERROR,"집계 불일치: payment_count, gross_amount, income_ta..."
2,OTHER,71,36375090,2800000,278000,70,36250090,2790000,278000,both,-1,-125000,-10000,0,ERROR,"집계 불일치: payment_count, gross_amount, income_tax"


In [24]:
all_events = pd.concat(
    [payment_events, rec_events],
    ignore_index=True
)

KEY_COLUMNS = ["scope", "key", "code"]

expected_set = set(
    truth[KEY_COLUMNS].itertuples(index=False, name=None)
)

detected_set = set(
    all_events[KEY_COLUMNS].itertuples(index=False, name=None)
)

tp = len(expected_set & detected_set)
fp = len(detected_set - expected_set)
fn = len(expected_set - detected_set)

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0

print("정답표 검증 항목:", len(expected_set))
print("탐지 항목:", len(detected_set))
print("정확한 탐지:", tp)
print("오탐:", fp)
print("미탐:", fn)
print(f"Precision: {precision:.2%}")
print(f"Recall: {recall:.2%}")

print("미탐 항목:", expected_set - detected_set)
print("오탐 항목:", detected_set - expected_set)

display(all_events)

정답표 검증 항목: 20
탐지 항목: 20
정확한 탐지: 20
오탐: 0
미탐: 0
Precision: 100.00%
Recall: 100.00%
미탐 항목: set()
오탐 항목: set()


/var/folders/6x/s8q88g217t9_ybc2846dwrj40000gn/T/ipykernel_35570/871966090.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_events = pd.concat(


,scope,key,code,status,message,expected,actual
0,PAYMENT,R001,DATA-01,ERROR,필수값 누락: recipient_id,NaN,None
1,PAYMENT,R002,DATA-03,ERROR,잘못된 금액: gross_amount,NaN,-100000
2,PAYMENT,R003,DATA-04,ERROR,유효하지 않은 지급일자,NaN,None
3,PAYMENT,R004,TAX-01,REVIEW,거주자 여부 추가 검토 필요,NaN,None
4,PAYMENT,R005,TAX-02,ERROR,소득세 불일치,30000.0,31000
5,PAYMENT,R006,TAX-03,ERROR,지방소득세 불일치,2400.0,2500
6,PAYMENT,R007,TAX-02,ERROR,소득세 불일치,0.0,10000
7,PAYMENT,R007,TAX-04,ERROR,기타소득 과세최저한 미적용,NaN,None
8,PAYMENT,R008,TAX-02,ERROR,소득세 불일치,900.0,0
9,PAYMENT,R008,TAX-03,ERROR,지방소득세 불일치,90.0,0


In [25]:
%%writefile src/report.py
from pathlib import Path

import pandas as pd
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

BASE_DIR = Path(__file__).resolve().parents[1]
OUTPUT_DIR = BASE_DIR / "outputs"

TAX_RULES = [
    [
        "INTEREST", "일반 과세 예금이자",
        "14%", "10%", "소액부징수 적용 제외",
        "소득세법 제86조, 제129조"
    ],
    [
        "BUSINESS", "계속적·반복적 인적용역",
        "3%", "10%", "소액부징수 적용 제외",
        "소득세법 제86조, 제129조; 시행령 제149조의3"
    ],
    [
        "OTHER", "일시적 강연료",
        "20%", "10%",
        "필요경비 60%, 소득금액 5만원 이하 과세최저한",
        "소득세법 제84조, 제129조; 시행령 제87조"
    ],
    [
        "LOCAL", "개인지방소득세",
        "-", "소득세의 10%", "특별징수",
        "지방세법 제103조의13"
    ],
    [
        "ROUNDING", "단수 처리",
        "-", "-", "지급 건별 10원 미만 절사 가정",
        "국고금 관리법 제47조; 지방회계법 제55조"
    ]
]


def evaluate_detection(events, truth):

    cols = ["scope", "key", "code"]

    expected = set(
        truth[cols].itertuples(index=False, name=None)
    )

    detected = set(
        events[cols].itertuples(index=False, name=None)
    )

    tp = len(expected & detected)
    fp = len(detected - expected)
    fn = len(expected - detected)

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0

    return {
        "expected_findings": len(expected),
        "detected_findings": len(detected),
        "true_positive": tp,
        "false_positive": fp,
        "false_negative": fn,
        "precision": precision,
        "recall": recall
    }


def create_report(
    transactions,
    events,
    detail_result,
    summary_result,
    truth
):

    OUTPUT_DIR.mkdir(exist_ok=True)

    output_path = OUTPUT_DIR / "validation_report.xlsx"

    metrics = evaluate_detection(events, truth)

    calculated = transactions.loc[
        transactions["tax_calculated"]
    ].copy()

    overview = pd.DataFrame([
        ("대상 지급월", "2026-10"),
        ("전체 지급자료 행 수", len(transactions)),
        ("원장 정상 행", int((transactions["status"] == "PASS").sum())),
        ("원장 오류 행", int((transactions["status"] == "ERROR").sum())),
        ("원장 추가 검토 행", int((transactions["status"] == "REVIEW").sum())),
        ("세액 비교 가능 행", len(calculated)),
        ("세액 비교 제외 행", len(transactions) - len(calculated)),
        ("거래별 대사 오류", int((detail_result["status"] == "ERROR").sum())),
        ("거래별 대사 보류", int((detail_result["status"] == "BLOCKED").sum())),
        ("집계 대사 오류", int((summary_result["status"] == "ERROR").sum())),
        ("소득세 과소징수", int(-calculated["income_tax_diff"].clip(upper=0).sum())),
        ("소득세 과다징수", int(calculated["income_tax_diff"].clip(lower=0).sum())),
        ("지방소득세 과소징수", int(-calculated["local_tax_diff"].clip(upper=0).sum())),
        ("지방소득세 과다징수", int(calculated["local_tax_diff"].clip(lower=0).sum())),
        ("검증 항목 탐지 수", metrics["detected_findings"]),
        ("오탐 수", metrics["false_positive"]),
        ("미탐 수", metrics["false_negative"]),
        ("Precision", f'{metrics["precision"]:.2%}'),
        ("Recall", f'{metrics["recall"]:.2%}'),
        ("범위", "가상 자료 및 지원 소득 유형에 한정"),
        ("주의", "실제 신고서가 아닌 신고 전 검증용 보고서")
    ], columns=["항목", "결과"])

    tax_summary = (
        calculated.groupby("income_type", as_index=False)
        .agg(
            payment_count=("payment_id", "size"),
            gross_amount=("gross_amount", "sum"),
            recorded_income_tax=("income_tax", "sum"),
            expected_income_tax=("expected_income_tax", "sum"),
            recorded_local_tax=("local_tax", "sum"),
            expected_local_tax=("expected_local_tax", "sum"),
            income_tax_diff=("income_tax_diff", "sum"),
            local_tax_diff=("local_tax_diff", "sum")
        )
    )

    tax_rules = pd.DataFrame(
        TAX_RULES,
        columns=[
            "구분", "적용 대상", "소득세율",
            "지방소득세율", "처리 기준", "법적 근거"
        ]
    )

    sheets = {
        "Summary": overview,
        "Tax Summary": tax_summary,
        "Transactions": transactions,
        "Errors": events[events["status"] == "ERROR"],
        "Review": events[events["status"] == "REVIEW"],
        "Reconciliation": detail_result,
        "Filing Summary": summary_result,
        "Tax Rules": tax_rules,
        "Ground Truth": truth
    }

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

        for sheet_name, frame in sheets.items():

            frame.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False
            )

            ws = writer.sheets[sheet_name]

            ws.freeze_panes = "A2"
            ws.auto_filter.ref = ws.dimensions

            for cell in ws[1]:

                cell.font = Font(
                    bold=True,
                    color="FFFFFF"
                )

                cell.fill = PatternFill(
                    "solid",
                    fgColor="203864"
                )

                cell.alignment = Alignment(
                    horizontal="center"
                )

            for column in ws.columns:

                width = max(
                    len(str(cell.value or ""))
                    for cell in column
                )

                ws.column_dimensions[
                    get_column_letter(column[0].column)
                ].width = min(max(width + 4, 15), 65)

            for row in ws.iter_rows(min_row=2):

                for cell in row:

                    if cell.value == "ERROR":

                        cell.fill = PatternFill(
                            "solid", fgColor="FCE8E6"
                        )
                        cell.font = Font(color="B91C1C")

                    elif cell.value in ["REVIEW", "BLOCKED"]:

                        cell.fill = PatternFill(
                            "solid", fgColor="FFF2CC"
                        )

                    elif cell.value == "PASS":

                        cell.fill = PatternFill(
                            "solid", fgColor="E2F0D9"
                        )

                    if isinstance(cell.value, (int, float)):
                        cell.number_format = "#,##0"

    return output_path, metrics

Writing src/report.py


In [26]:
from src.report import create_report

report_path, metrics = create_report(
    transactions,
    all_events,
    detail_result,
    summary_result,
    truth
)

print(report_path)
print(metrics)

/Users/kosunjin/Desktop/TaxDataValidator/outputs/validation_report.xlsx
{'expected_findings': 20, 'detected_findings': 20, 'true_positive': 20, 'false_positive': 0, 'false_negative': 0, 'precision': 1.0, 'recall': 1.0}


In [27]:
%%writefile tests/test_tax_rules.py
import pytest

from src.tax_rules import (
    calculate_tax,
    truncate_10,
    ReviewRequired,
    SUBTYPES
)


@pytest.mark.parametrize(
    "kind,gross,expected_tax,expected_local",
    [
        ("INTEREST", 1000000, 140000, 14000),
        ("INTEREST", 5000, 700, 70),
        ("INTEREST", 10000, 1400, 140),
        ("INTEREST", 200000, 28000, 2800),
        ("BUSINESS", 1000000, 30000, 3000),
        ("BUSINESS", 30000, 900, 90),
        ("BUSINESS", 20000, 600, 60),
        ("BUSINESS", 800000, 24000, 2400),
        ("OTHER", 1000000, 80000, 8000),
        ("OTHER", 500000, 40000, 4000),
        ("OTHER", 120000, 0, 0),
        ("OTHER", 124995, 0, 0),
        ("OTHER", 125000, 0, 0),
        ("OTHER", 125005, 10000, 1000),
        ("OTHER", 125100, 10000, 1000),
        ("OTHER", 125125, 10010, 1000),
        ("BUSINESS", 33300, 990, 90),
        ("BUSINESS", 33400, 1000, 100),
        ("INTEREST", 7142, 990, 90),
        ("INTEREST", 7143, 1000, 100)
    ]
)
def test_calculate_tax(
    kind, gross, expected_tax, expected_local
):

    result = calculate_tax(
        kind,
        gross,
        income_subtype=SUBTYPES[kind]
    )

    assert result["income_tax"] == expected_tax
    assert result["local_tax"] == expected_local

    assert result["net_amount"] == (
        gross - expected_tax - expected_local
    )


@pytest.mark.parametrize(
    "amount,expected",
    [
        (1234, 1230),
        (1238, 1230),
        (999, 990),
        (9, 0),
        (1000, 1000)
    ]
)
def test_truncate_10(amount, expected):

    assert truncate_10(amount) == expected


@pytest.mark.parametrize(
    "gross",
    [-100000, 0, None, "100000", 100000.5, True]
)
def test_invalid_amount(gross):

    with pytest.raises(ValueError):
        calculate_tax(
            "BUSINESS",
            gross,
            income_subtype="RECURRING_SERVICE"
        )


def test_non_resident():

    with pytest.raises(ReviewRequired):
        calculate_tax(
            "INTEREST",
            1000000,
            "NON_RESIDENT",
            "BANK_DEPOSIT"
        )


def test_unsupported_income():

    with pytest.raises(ReviewRequired):
        calculate_tax("SALARY", 3000000)


def test_unsupported_subtype():

    with pytest.raises(ReviewRequired):
        calculate_tax(
            "INTEREST",
            1000000,
            income_subtype="TAX_EXEMPT"
        )


def test_missing_subtype():

    with pytest.raises(ReviewRequired):
        calculate_tax("OTHER", 1000000)


def test_fractional_expense_review():

    with pytest.raises(ReviewRequired):
        calculate_tax(
            "OTHER",
            125001,
            income_subtype="ONE_TIME_LECTURE"
        )


def test_expense_amount():

    result = calculate_tax(
        "OTHER",
        500000,
        income_subtype="ONE_TIME_LECTURE"
    )

    assert result["expense_amount"] == 300000
    assert result["tax_base"] == 200000

Writing tests/test_tax_rules.py


In [28]:
%%writefile tests/test_validation.py
import pandas as pd
import pytest

from src.generate_data import generate_data
from src.validator import validate_payments
from src.reconciliation import reconcile
from src.report import evaluate_detection


@pytest.fixture(scope="module")
def sample():

    payments, details, summary, truth = generate_data()

    transactions, payment_events = validate_payments(payments)

    detail_result, summary_result, rec_events = reconcile(
        transactions,
        details,
        summary
    )

    events = pd.concat(
        [payment_events, rec_events],
        ignore_index=True
    )

    return {
        "payments": payments,
        "details": details,
        "summary": summary,
        "truth": truth,
        "transactions": transactions,
        "events": events,
        "detail_result": detail_result,
        "summary_result": summary_result
    }


@pytest.mark.parametrize(
    "row_id,code",
    [
        ("R001", "DATA-01"),
        ("R002", "DATA-03"),
        ("R003", "DATA-04"),
        ("R004", "TAX-01"),
        ("R005", "TAX-02"),
        ("R006", "TAX-03"),
        ("R007", "TAX-04"),
        ("R008", "TAX-05"),
        ("R009", "TAX-06"),
        ("R010", "DATA-02"),
        ("R200", "DATA-02")
    ]
)
def test_payment_errors(sample, row_id, code):

    events = sample["events"]

    assert (
        (events["scope"] == "PAYMENT")
        & (events["key"] == row_id)
        & (events["code"] == code)
    ).any()


@pytest.mark.parametrize(
    "key,code",
    [
        ("P011", "REC-01"),
        ("P012", "REC-03"),
        ("P999", "REC-02"),
        ("INTEREST", "REC-04"),
        ("BUSINESS", "REC-04"),
        ("OTHER", "REC-04")
    ]
)
def test_reconciliation(sample, key, code):

    events = sample["events"]

    assert (
        (events["key"] == key)
        & (events["code"] == code)
    ).any()


def test_dataset_size(sample):

    assert len(sample["payments"]) == 200


def test_normal_transactions(sample):

    assert (
        sample["transactions"]["status"] == "PASS"
    ).sum() == 189


def test_review_status(sample):

    row = sample["transactions"].loc[
        sample["transactions"]["row_id"] == "R004"
    ].iloc[0]

    assert row["status"] == "REVIEW"
    assert not row["tax_calculated"]


def test_duplicate_exclusion(sample):

    rows = sample["transactions"].loc[
        sample["transactions"]["payment_id"] == "P010"
    ]

    assert len(rows) == 2
    assert not rows["reconciliation_eligible"].any()


def test_missing_column(sample):

    frame = sample["payments"].drop(
        columns=["gross_amount"]
    )

    with pytest.raises(ValueError):
        validate_payments(frame)


def test_invalid_date(sample):

    frame = sample["payments"].iloc[[15]].copy()
    frame.loc[:, "payment_date"] = "2026-11-01"

    _, events = validate_payments(frame)

    assert "DATA-04" in events["code"].values


def test_expense_review(sample):

    frame = sample["payments"].iloc[[6]].copy()

    frame.loc[:, "expense_amount"] = 90000
    frame.loc[:, "income_tax"] = 0
    frame.loc[:, "net_amount"] = 125000

    transactions, events = validate_payments(frame)

    assert transactions.iloc[0]["status"] == "REVIEW"
    assert "TAX-07" in events["code"].values


def test_duplicate_filing(sample):

    details = pd.concat(
        [
            sample["details"],
            sample["details"].iloc[[20]]
        ],
        ignore_index=True
    )

    _, summary_result, events = reconcile(
        sample["transactions"],
        details,
        sample["summary"]
    )

    assert "DATA-02" in events["code"].values
    assert "BLOCKED" in summary_result["status"].values


def test_filing_invalid_amount(sample):

    details = sample["details"].copy()
    details.loc[20, "gross_amount"] = -50000

    _, _, events = reconcile(
        sample["transactions"],
        details,
        sample["summary"]
    )

    assert "DATA-03" in events["code"].values


def test_detection_precision(sample):

    metrics = evaluate_detection(
        sample["events"],
        sample["truth"]
    )

    assert metrics["precision"] == 1.0


def test_detection_recall(sample):

    metrics = evaluate_detection(
        sample["events"],
        sample["truth"]
    )

    assert metrics["recall"] == 1.0


def test_reproducibility(sample):

    payments, details, summary, truth = generate_data(seed=42)

    pd.testing.assert_frame_equal(
        sample["payments"],
        payments
    )

    pd.testing.assert_frame_equal(
        sample["details"],
        details
    )

    pd.testing.assert_frame_equal(
        sample["summary"],
        summary
    )

Writing tests/test_validation.py


In [29]:
import sys
import subprocess

test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-q"],
    capture_output=True,
    text=True
)

print(test_result.stdout)
print(test_result.stderr)

assert test_result.returncode == 0, "테스트 실패"

..................................................................       [100%]
=============================== warnings summary ===============================
../../../../opt/miniconda3/envs/aice/lib/python3.12/site-packages/anyio/pytest_plugin.py:14
  /opt/miniconda3/envs/aice/lib/python3.12/site-packages/anyio/pytest_plugin.py:14: PytestRemovedIn10Warning: _pytest.python.CallSpec2 has been renamed to CallSpec.
  The CallSpec2 alias will be removed in pytest 10.
  Update imports to use CallSpec instead.
  See https://docs.pytest.org/en/stable/deprecations.html#callspec2-renamed
    from _pytest.python import CallSpec2

tests/test_validation.py::test_payment_errors[R001-DATA-01]
  /Users/kosunjin/Desktop/TaxDataValidator/tests/test_validation.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, ex

In [30]:
%%writefile main.py
from pathlib import Path
from time import perf_counter
import argparse

import pandas as pd

from src.generate_data import generate_data
from src.validator import validate_payments
from src.reconciliation import reconcile
from src.report import create_report

BASE_DIR = Path(__file__).resolve().parent
DATA_DIR = BASE_DIR / "data"


def main(regenerate=True):

    start = perf_counter()

    if regenerate:
        generate_data(seed=42)

    payments = pd.read_csv(
        DATA_DIR / "payments.csv",
        keep_default_na=False
    )

    details = pd.read_csv(
        DATA_DIR / "filing_details.csv",
        keep_default_na=False
    )

    summary = pd.read_csv(
        DATA_DIR / "filing_summary.csv",
        keep_default_na=False
    )

    truth = pd.read_csv(
        DATA_DIR / "ground_truth.csv",
        keep_default_na=False
    )

    transactions, payment_events = validate_payments(payments)

    detail_result, summary_result, rec_events = reconcile(
        transactions,
        details,
        summary
    )

    events = pd.concat(
        [payment_events, rec_events],
        ignore_index=True
    )

    report_path, metrics = create_report(
        transactions,
        events,
        detail_result,
        summary_result,
        truth
    )

    elapsed = perf_counter() - start

    print("=" * 50)
    print("TaxDataValidator 1.0")
    print("=" * 50)

    print(f"지급자료: {len(payments)}건")
    print(f"정상 행: {(transactions['status'] == 'PASS').sum()}건")
    print(f"오류 행: {(transactions['status'] == 'ERROR').sum()}건")
    print(f"검토 행: {(transactions['status'] == 'REVIEW').sum()}건")

    print("-" * 50)

    print(f"탐지 항목: {metrics['detected_findings']}건")
    print(f"오탐: {metrics['false_positive']}건")
    print(f"미탐: {metrics['false_negative']}건")
    print(f"Precision: {metrics['precision']:.2%}")
    print(f"Recall: {metrics['recall']:.2%}")

    print("-" * 50)

    print(f"실행시간: {elapsed:.4f}초")
    print(f"보고서: {report_path}")

    return elapsed, metrics


if __name__ == "__main__":

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--reuse-data",
        action="store_true"
    )

    args = parser.parse_args()

    main(regenerate=not args.reuse_data)

Writing main.py


In [31]:
%run main.py

/Users/kosunjin/Desktop/TaxDataValidator/main.py:51: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat(


TaxDataValidator 1.0
지급자료: 200건
정상 행: 189건
오류 행: 10건
검토 행: 1건
--------------------------------------------------
탐지 항목: 20건
오탐: 0건
미탐: 0건
Precision: 100.00%
Recall: 100.00%
--------------------------------------------------
실행시간: 0.7277초
보고서: /Users/kosunjin/Desktop/TaxDataValidator/outputs/validation_report.xlsx


In [32]:
from main import main
import statistics
import pandas as pd

benchmark = []

for i in range(5):

    elapsed, result = main()

    benchmark.append({
        "run": i + 1,
        "elapsed_seconds": round(elapsed, 4),
        "precision": result["precision"],
        "recall": result["recall"],
        "false_positive": result["false_positive"],
        "false_negative": result["false_negative"]
    })

benchmark_df = pd.DataFrame(benchmark)

benchmark_df.to_csv(
    "outputs/benchmark.csv",
    index=False
)

display(benchmark_df)

print(
    "평균 실행시간:",
    round(statistics.mean(
        benchmark_df["elapsed_seconds"]
    ), 4)
)

print(
    "중앙값 실행시간:",
    round(statistics.median(
        benchmark_df["elapsed_seconds"]
    ), 4)
)

print(
    "전체 검증 성공:",
    bool(
        (benchmark_df["precision"] == 1).all()
        and (benchmark_df["recall"] == 1).all()
    )
)

/Users/kosunjin/Desktop/TaxDataValidator/main.py:51: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat(


TaxDataValidator 1.0
지급자료: 200건
정상 행: 189건
오류 행: 10건
검토 행: 1건
--------------------------------------------------
탐지 항목: 20건
오탐: 0건
미탐: 0건
Precision: 100.00%
Recall: 100.00%
--------------------------------------------------
실행시간: 0.6883초
보고서: /Users/kosunjin/Desktop/TaxDataValidator/outputs/validation_report.xlsx


/Users/kosunjin/Desktop/TaxDataValidator/main.py:51: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat(


TaxDataValidator 1.0
지급자료: 200건
정상 행: 189건
오류 행: 10건
검토 행: 1건
--------------------------------------------------
탐지 항목: 20건
오탐: 0건
미탐: 0건
Precision: 100.00%
Recall: 100.00%
--------------------------------------------------
실행시간: 0.7434초
보고서: /Users/kosunjin/Desktop/TaxDataValidator/outputs/validation_report.xlsx


/Users/kosunjin/Desktop/TaxDataValidator/main.py:51: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat(


TaxDataValidator 1.0
지급자료: 200건
정상 행: 189건
오류 행: 10건
검토 행: 1건
--------------------------------------------------
탐지 항목: 20건
오탐: 0건
미탐: 0건
Precision: 100.00%
Recall: 100.00%
--------------------------------------------------
실행시간: 0.5537초
보고서: /Users/kosunjin/Desktop/TaxDataValidator/outputs/validation_report.xlsx


... (2 outputs left out)

/Users/kosunjin/Desktop/TaxDataValidator/main.py:51: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat(


TaxDataValidator 1.0
지급자료: 200건
정상 행: 189건
오류 행: 10건
검토 행: 1건
--------------------------------------------------
탐지 항목: 20건
오탐: 0건
미탐: 0건
Precision: 100.00%
Recall: 100.00%
--------------------------------------------------
실행시간: 1.8976초
보고서: /Users/kosunjin/Desktop/TaxDataValidator/outputs/validation_report.xlsx


,run,elapsed_seconds,precision,recall,false_positive,false_negative
0,1,0.6883,1.0,1.0,0,0
1,2,0.7434,1.0,1.0,0,0
2,3,0.5537,1.0,1.0,0,0
3,4,1.2000,1.0,1.0,0,0
4,5,1.8976,1.0,1.0,0,0


평균 실행시간: 1.0166
중앙값 실행시간: 0.7434
전체 검증 성공: True


In [33]:
%%writefile docs/tax_rules.md
# Tax Rules and Scope

기준일: 2026년 11월 24일
대상 지급월: 2026년 10월

## 1. 적용 세율

| 소득 유형 | 소득세 | 지방소득세 |
|---|---:|---:|
| 일반 과세 예금이자 | 14% | 소득세의 10% |
| 계속적·반복적 인적용역 사업소득 | 3% | 소득세의 10% |
| 일시적 강연료 기타소득 | 소득금액의 20% | 소득세의 10% |

## 2. 법적 근거

- 소득세법 제84조: 기타소득 과세최저한
- 소득세법 제86조: 소액 부징수
- 소득세법 제127조: 원천징수의무
- 소득세법 제128조: 원천징수세액의 납부
- 소득세법 제129조: 원천징수세율
- 소득세법 시행령 제87조: 기타소득 필요경비 계산
- 소득세법 시행령 제149조의3: 소액 부징수의 예외
- 지방세법 제103조의13: 특별징수의무
- 국고금 관리법 제47조: 끝수 계산
- 지방회계법 제55조: 끝수 계산

## 3. 공식 자료

- 소득세법: https://www.law.go.kr/법령/소득세법
- 소득세법 시행령: https://www.law.go.kr/법령/소득세법시행령
- 지방세법: https://www.law.go.kr/법령/지방세법
- 국고금 관리법: https://www.law.go.kr/법령/국고금관리법
- 지방회계법: https://www.law.go.kr/법령/지방회계법
- 국세청 원천징수 세율: https://nts.go.kr/nts/cm/cntnts/cntntsView.do?cntntsId=7703&mi=2291
- 국세청 기타소득 안내: https://www.nts.go.kr/nts/cm/cntnts/cntntsView.do?cntntsId=7893

## 4. 적용 가정

- 국내 거주자인 개인에게 지급한 소득
- 일반 과세 예금이자
- 소액부징수 예외에 해당하는 인적용역 사업소득
- 필요경비 60%가 적용되는 일시적 강연료
- 실제 필요경비가 인정되는 60%를 초과하지 않는 경우
- 지급 행마다 독립적인 소득 발생 건을 가정
- 세액의 지급 건별 10원 미만 절사는 학습용 가정
- 원 미만 필요경비 발생 건은 추가 검토 대상으로 분류

## 5. 한계

- 실제 소득 유형 분류는 계약 및 사실관계 확인 필요
- 비거주자, 조세조약, 근로소득, 비과세 및 과세특례 미지원
- 실제 단수 처리의 적용 단위 및 시점 추가 검증 필요
- 법정 신고서와 지급명세서 전자신고 파일 미생성
- 홈택스 및 위택스 연동 미지원
- 지급명세서 제출 의무 및 제출기한 자동 판정 미지원
- 모든 데이터는 가상 데이터
- 지급 원장과 신고자료는 공통 가상 기록에서 생성 후
  서로 다른 오류를 삽입한 시뮬레이션 자료
- 탐지 성과는 설계된 가상 오류에 한정
- 실제 회사 업무의 오류 탐지 성능이나 시간 절감 효과를
  입증하는 결과가 아님

Writing docs/tax_rules.md


In [34]:
from importlib.metadata import version
from pathlib import Path

packages = [
    "pandas",
    "numpy",
    "openpyxl",
    "pytest"
]

requirements = "\n".join(
    f"{package}=={version(package)}"
    for package in packages
)

Path("requirements.txt").write_text(
    requirements + "\n",
    encoding="utf-8"
)

print(requirements)

pandas==2.3.1
numpy==1.26.4
openpyxl==3.1.5
pytest==9.2.0


In [35]:
%%writefile .gitignore
__pycache__/
*.pyc
.ipynb_checkpoints/
.pytest_cache/
.coverage
.DS_Store
.venv/
venv/
.env
*.log
~$*.xlsx

Writing .gitignore


In [36]:
%%writefile README.md
# TaxDataValidator 1.0

Python-based Withholding Tax Data Validation System

## 프로젝트 소개

기업의 원천세 신고 과정에서 발생할 수 있는 세액 계산 오류,
데이터 누락 및 중복, 신고자료 불일치를 탐지하기 위한
규칙 기반 세무 데이터 검증 프로젝트입니다.

## 개발 목적

- 원천징수세액 계산 및 검증 자동화
- 신고 기초자료의 정합성 확보
- 지급 원장과 신고자료 간 대사 자동화
- 세무 담당자의 추가 검토 대상 식별
- 검증 결과의 Excel 보고서 생성

## 주요 기능

| 기능 | 구현 내용 |
|---|---|
| Synthetic Data | 재현 가능한 가상 지급자료 200행 생성 |
| Tax Calculation | 이자·사업·기타소득 원천징수세액 재계산 |
| Tax Validation | 과세최저한 및 소액부징수 예외 검증 |
| Data Validation | 필수값 누락, 중복, 금액 및 날짜 오류 탐지 |
| Reconciliation | 지급 원장과 신고 상세·집계자료 대사 |
| Exception Handling | 오류 및 추가 검토 대상 분류 |
| Reporting | 9개 시트로 구성된 Excel 보고서 생성 |
| Unit Testing | 정상·오류·경계값·예외 테스트 |
| Benchmark | 반복 실행 성능 및 재현성 측정 |

## 기술 스택

- Python
- Pandas
- NumPy
- Openpyxl
- Pytest
- JupyterLab
- Git / GitHub

## 프로젝트 구조

    TaxDataValidator/
    ├── data/
    │   ├── payments.csv
    │   ├── filing_details.csv
    │   ├── filing_summary.csv
    │   └── ground_truth.csv
    ├── src/
    │   ├── __init__.py
    │   ├── generate_data.py
    │   ├── tax_rules.py
    │   ├── validator.py
    │   ├── reconciliation.py
    │   └── report.py
    ├── tests/
    │   ├── test_tax_rules.py
    │   └── test_validation.py
    ├── docs/
    │   └── tax_rules.md
    ├── outputs/
    │   ├── validation_report.xlsx
    │   └── benchmark.csv
    ├── notebooks/
    ├── main.py
    ├── requirements.txt
    └── README.md

## 실행 방법

    pip install -r requirements.txt
    python main.py

기존 가상 데이터 재사용:

    python main.py --reuse-data

단위 테스트:

    python -m pytest tests -q

## 검증 상태

- PASS: 구현된 검증 규칙상 이상 없음
- ERROR: 데이터 또는 계산상 오류
- REVIEW: 추가 세무 검토 필요
- BLOCKED: 정합성 오류로 대사 보류

## 검증 규칙

- DATA-01~04: 데이터 정합성 검증
- TAX-01~07: 원천징수세액 및 세무 판단 검증
- REC-01~04: 신고자료 대사

## 결과물

- [Excel 검증 보고서](outputs/validation_report.xlsx)
- [반복 실행 결과](outputs/benchmark.csv)
- [세법 근거 및 적용 범위](docs/tax_rules.md)

## 정확성 평가

사전에 수기로 계산한 세액 사례와 독립적으로 지정한
오류 정답표를 기준으로 Precision과 Recall을 측정합니다.

해당 지표는 가상 데이터에 삽입한 오류에 대한 탐지 성능이며,
실제 기업의 세무 업무 정확도를 의미하지 않습니다.

## 적용 범위 및 한계

본 프로젝트는 국내 거주자인 개인의 일반 과세 예금이자,
특정 인적용역 사업소득 및 일시적 강연료를 대상으로 합니다.

실제 세무 신고 프로그램이 아닌 학습용 검증 시뮬레이션이며,
비거주자 과세, 근로소득, 조세특례, 실제 전자신고 등은
지원 범위에 포함되지 않습니다.

## Disclaimer

All datasets are synthetic.
This project is intended for educational and portfolio purposes.
It is not a production tax filing system.

Writing README.md


In [37]:
import re
import subprocess
import sys
from pathlib import Path
import pandas as pd

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-q"],
    capture_output=True,
    text=True
)

print(result.stdout)

assert result.returncode == 0, result.stdout

match = re.search(r"(\d+) passed", result.stdout)

assert match is not None, "테스트 통과 수 확인 필요"

test_count = int(match.group(1))

benchmark = pd.read_csv("outputs/benchmark.csv")

performance = f"""
## 실행 및 검증 결과

| 평가 항목 | 결과 |
|---|---:|
| 가상 지급자료 | 200행 |
| 검증 규칙 | 15개 |
| 단위 테스트 통과 | {test_count}개 |
| 반복 실행 | {len(benchmark)}회 |
| 평균 실행시간 | {benchmark["elapsed_seconds"].mean():.4f}초 |
| 중앙값 실행시간 | {benchmark["elapsed_seconds"].median():.4f}초 |
| 평균 Precision | {benchmark["precision"].mean():.2%} |
| 평균 Recall | {benchmark["recall"].mean():.2%} |
| 오탐 합계 | {int(benchmark["false_positive"].sum())}건 |
| 미탐 합계 | {int(benchmark["false_negative"].sum())}건 |

실행환경: MacBook Pro 2017 Intel, JupyterLab

측정 시간에는 가상 데이터 생성, CSV 입출력,
세액 검증, 신고자료 대사 및 Excel 보고서 생성이 포함됩니다.
"""

path = Path("README.md")
content = path.read_text(encoding="utf-8")

content = content.split("\n## 실행 및 검증 결과")[0]

path.write_text(
    content.rstrip() + "\n\n" + performance,
    encoding="utf-8"
)

print(performance)

..................................................................       [100%]
=============================== warnings summary ===============================
../../../../opt/miniconda3/envs/aice/lib/python3.12/site-packages/anyio/pytest_plugin.py:14
  /opt/miniconda3/envs/aice/lib/python3.12/site-packages/anyio/pytest_plugin.py:14: PytestRemovedIn10Warning: _pytest.python.CallSpec2 has been renamed to CallSpec.
  The CallSpec2 alias will be removed in pytest 10.
  Update imports to use CallSpec instead.
  See https://docs.pytest.org/en/stable/deprecations.html#callspec2-renamed
    from _pytest.python import CallSpec2

tests/test_validation.py::test_payment_errors[R001-DATA-01]
  /Users/kosunjin/Desktop/TaxDataValidator/tests/test_validation.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, ex

In [38]:
from pathlib import Path

required_files = [
    "main.py",
    "README.md",
    "requirements.txt",
    ".gitignore",
    "docs/tax_rules.md",
    "src/generate_data.py",
    "src/tax_rules.py",
    "src/validator.py",
    "src/reconciliation.py",
    "src/report.py",
    "tests/test_tax_rules.py",
    "tests/test_validation.py",
    "data/payments.csv",
    "data/filing_details.csv",
    "data/filing_summary.csv",
    "data/ground_truth.csv",
    "outputs/validation_report.xlsx",
    "outputs/benchmark.csv"
]

for filename in required_files:

    exists = Path(filename).is_file()

    print(
        "✅" if exists else "❌",
        filename
    )

assert all(
    Path(f).is_file()
    for f in required_files
)

print("\nTaxDataValidator 1.0 파일 생성 완료")

✅ main.py
✅ README.md
✅ requirements.txt
✅ .gitignore
✅ docs/tax_rules.md
✅ src/generate_data.py
✅ src/tax_rules.py
✅ src/validator.py
✅ src/reconciliation.py
✅ src/report.py
✅ tests/test_tax_rules.py
✅ tests/test_validation.py
✅ data/payments.csv
✅ data/filing_details.csv
✅ data/filing_summary.csv
✅ data/ground_truth.csv
✅ outputs/validation_report.xlsx
✅ outputs/benchmark.csv

TaxDataValidator 1.0 파일 생성 완료


In [39]:
print(performance)


## 실행 및 검증 결과

| 평가 항목 | 결과 |
|---|---:|
| 가상 지급자료 | 200행 |
| 검증 규칙 | 15개 |
| 단위 테스트 통과 | 66개 |
| 반복 실행 | 5회 |
| 평균 실행시간 | 1.0166초 |
| 중앙값 실행시간 | 0.7434초 |
| 평균 Precision | 100.00% |
| 평균 Recall | 100.00% |
| 오탐 합계 | 0건 |
| 미탐 합계 | 0건 |

실행환경: MacBook Pro 2017 Intel, JupyterLab

측정 시간에는 가상 데이터 생성, CSV 입출력,
세액 검증, 신고자료 대사 및 Excel 보고서 생성이 포함됩니다.

